# Dealing with Counts: Binarization

_Feature Engineering_

**Sometimes only presence matters, not how much — clip a raw count to 0/1.**

A count is a non-negative integer: how many times something happened. Counts are everywhere — plays, clicks, words, visits.

       The trap: counts can be wildly skewed. A few items get astronomically high counts; most get tiny ones. Plotted, the distribution has a long right tail. Fed to a model, the giant values dominate.

---

This notebook is a practice scaffold for the **Dealing with Counts: Binarization** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## See it for yourself: the problem, then the fix

Run this cell. It plots the **raw** data and reproduces the problem it causes, then plots the **engineered** data and shows the fix — with before/after numbers.

In [ ]:
# Lesson: Binarization — collapse a skewed count to a 0/1 flag.
#
# PROBLEM: a right-skewed count feature has a few HUGE values. Standardizing it
# (subtract mean / divide by std) lets those outliers stretch the scale to +10 std,
# so in a distance model (k-NN) that one column DOMINATES and drowns out good features.
# FIX: binarize to 0/1 ("did it happen / is it big"). The flag is bounded, outlier-free,
# and for many count features the signal IS "crossed a threshold" — so it keeps signal.
#
# We show the raw problem on REAL data (breast-cancer 'mean area'), then the predictive
# fix on a small ILLUSTRATIVE count feature (fixed rng) — honest note: the bundled
# feature is too clean to hurt a 1-D k-NN by itself, so the win is shown where it's real.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# ---- STEP 1: Load real data --------------------------------------------------
# Breast-cancer 'mean area': real, heavily right-skewed (most tumors small, a few huge).
data = load_breast_cancer()
area = data.data[:, data.feature_names.tolist().index("mean area")]

# ---- STEP 2: Visualize the PURE (raw) data -----------------------------------
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].hist(area, bins=40, color="#c0392b")
ax[0].axvline(np.median(area), color="black", ls="--", label="median")
ax[0].set_title(f"STEP 2: raw 'mean area' (right-skewed)\nmax={area.max():.0f} = "
                f"{area.max()/np.median(area):.1f}x the median")
ax[0].set_xlabel("mean area"); ax[0].set_ylabel("count"); ax[0].legend()

# ---- STEP 3: Reproduce the PROBLEM — outliers dominate the standardized range -
# Standardize the raw feature. A well-behaved feature lands roughly in [-3, +3].
# Here the long right tail throws the top values WAY past that: the scale is hijacked
# by a few points, so in a distance model this column would dominate everything else.
z = StandardScaler().fit_transform(area.reshape(-1, 1)).ravel()
print(f"STEP 3 PROBLEM  raw standardized range = [{z.min():.1f}, {z.max():.1f}]")
print(f"  -> {(z > 3).sum()} points sit beyond +3 std (top point at +{z.max():.1f}); "
      f"a healthy feature would stop near +3. The tail hijacks the scale.")

# ---- STEP 4: Binarize, then visualize the engineered feature -----------------
# Threshold the REAL feature at its median -> 0/1. Now standardized it is bounded.
flag = (area > np.median(area)).astype(int)
zb = StandardScaler().fit_transform(flag.reshape(-1, 1)).ravel()
print(f"STEP 4 FIX      binarized standardized range = [{zb.min():.1f}, {zb.max():.1f}] "
      f"(bounded, outlier-free)")
ax[1].hist(z, bins=40, color="#c0392b", alpha=.7, label="raw (tail to +5)")
ax[1].hist(zb, bins=40, color="#27ae60", alpha=.7, label="binarized (bounded)")
ax[1].set_title("STEP 4: standardized RANGE\nraw blows out vs binarized stays bounded")
ax[1].set_xlabel("standardized value"); ax[1].set_ylabel("count"); ax[1].legend()

# ---- STEP 5: Show the FIX helps a real model ---------------------------------
# ILLUSTRATIVE dataset (fixed rng, labeled as such): a clean informative feature
# 'good' PLUS a skewed COUNT feature whose true signal is "happened at all" but whose
# raw magnitude has a heavy lognormal tail. The raw count's outliers dominate the k-NN
# distance and drag accuracy DOWN; the 0/1 flag fixes it. This mirrors the real
# 'mean area' skew above in a setting where the predictive damage is visible.
rng = np.random.default_rng(0)
n = 600
y = rng.integers(0, 2, n)                              # labels
good = y + rng.normal(0, 0.6, n)                       # clean, informative feature
happened = ((y * 0.8 + rng.random(n)) > 0.6).astype(int)   # the TRUE binary signal
count = happened * rng.lognormal(2.0, 1.8, n)          # heavy-tailed raw magnitude

def knn_auc(X):  # same model, same CV everywhere -> honest comparison
    pipe = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15))
    return cross_val_score(pipe, X, y, cv=5, scoring="roc_auc").mean()

X_raw = np.column_stack([good, count])                 # good + raw skewed count
X_bin = np.column_stack([good, (count > 0).astype(int)])  # good + 0/1 flag
raw_auc = knn_auc(X_raw)
eng_auc = knn_auc(X_bin)

# class balance per half of the flag — shows the flag carries real signal
for f in (0, 1):
    m = (count > 0).astype(int) == f
    print(f"  flag={f} ('{'big' if f else 'zero'}'): n={m.sum():3d}  "
          f"positive rate = {y[m].mean():.2f}")

# bar of the 0/1 split stacked by class
fl = (count > 0).astype(int)
z0 = ((fl == 0) & (y == 0)).sum(); o0 = ((fl == 0) & (y == 1)).sum()
z1 = ((fl == 1) & (y == 0)).sum(); o1 = ((fl == 1) & (y == 1)).sum()
ax[2].bar([0, 1], [z0, z1], color="#2980b9", label="class 0")
ax[2].bar([0, 1], [o0, o1], bottom=[z0, z1], color="#c0392b", label="class 1")
ax[2].set_xticks([0, 1]); ax[2].set_xticklabels(["flag=0", "flag=1"])
ax[2].set_title("STEP 5: 0/1 split (illustrative)\nclass balance per half"); ax[2].legend()
plt.tight_layout(); plt.show()

print(f"STEP 5  k-NN with raw skewed count   ROC-AUC = {raw_auc:.3f}")
print(f"STEP 5  k-NN with binarized 0/1 flag ROC-AUC = {eng_auc:.3f}")
print(f"PROBLEM (raw): {raw_auc:.3f}   ->   FIX (binarized): {eng_auc:.3f}")

## Reference implementation — pandas

In [ ]:
import pandas as pd
import numpy as np

# Echo Nest taste profile subset (Million Song Dataset).
# Columns: user (hash), song (hash), listen_count (int).
# Download: github.com/alicezheng/feature-engineering-book  (and the MSD/Echo Nest site)
listen_count = pd.read_csv(
    'train_triplets.txt', sep='\t', header=None,
    names=['user', 'song', 'listen_count']
)

# ----- the raw counts are heavily skewed -----
print(listen_count['listen_count'].describe())
# one user can play a single song hundreds of times, while most plays are 1-2:
#   mean  ~ 3, max in the hundreds -> a long right tail

# ----- BINARIZE: for a recommender we only care WHETHER a user listened -----
# clip every positive play count to 1; absence stays 0.
listen_count['listen'] = (listen_count['listen_count'] >= 1).astype(int)

# equivalently, with scikit-learn's Binarizer (threshold = 0 keeps >0 as 1):
# from sklearn.preprocessing import binarize
# listen_count['listen'] = binarize(listen_count[['listen_count']], threshold=0.0)

print(listen_count[['listen_count', 'listen']].head(10))
#    listen_count  listen
# 0             1       1
# 1             2       1
# 2             1       1
# 3           312       1     <- the super-fan's count collapses to 1
# 4             1       1

# The 'listen' column is the robust presence/absence signal the book feeds
# to the recommender, instead of the outlier-prone raw 'listen_count'.

## Visualize the data & results

_On a real right-skewed count-like feature, what does the raw distribution look like, and how does binarizing at the median split the classes?_

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()                 # 569 real tumor records
names = list(data.feature_names)
X, y = data.data, data.target               # label: 1 = benign, 0 = malignant

# pick a real count-like, right-skewed feature: 'mean area'
col = X[:, names.index('mean area')]
print("min/median/max =", round(col.min(),1), round(np.median(col),1), round(col.max(),1))
# -> 143.5 / 551.1 / 2501.0  (a long right tail)
skew = ((col - col.mean())**3).mean() / col.std()**3
print("skew =", round(float(skew), 2))      # -> 1.64, heavily right-skewed

# raw distribution: histogram counts pile up on the left
counts, edges = np.histogram(col, bins=8)
print("hist counts =", counts.tolist())     # -> [164, 249, 69, 61, 14, 8, 1, 3]

# BINARIZE at the median (no natural zero, so split the rows in half)
thr = np.median(col)
b = (col >= thr).astype(int)
print("n(0), n(1) =", int((b == 0).sum()), int((b == 1).sum()))   # -> 284, 285

# is the 0/1 split informative? class balance within each half
for v in (0, 1):
    sel = b == v
    print("bin", v, "-> P(benign) =", round(float(y[sel].mean()), 3))
# bin 0 -> 0.940   (small-area half is mostly benign)
# bin 1 -> 0.316   (large-area half is mostly malignant)

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** You have a "listen_count" column from the Echo Nest data: most values are 1 or 2, but one user played a song 980 times. You are building a song recommender. Should you feed the raw count, and what does binarizing change?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Ask what the recommender actually needs from this column. — _The training signal is "this user likes this song." Presence of an interaction is the weak positive; the exact replay count is mostly noise._
- Notice the 980 is a power-user artifact. — _A heavy-tailed count lets one row dominate a linear model's loss and distort coefficients — the value is about that user's habit, not preference strength._
- Apply b = 1[x >= 1] to collapse every positive count to 1. — _Binarizing maps 1, 2, and 980 all to 1, removing the skew and the outlier's leverage. The feature becomes "listened at all"._

**Answer:** Don't feed the raw count: it is heavy-tailed and the 980 would dominate. Binarize with threshold 1 — the column becomes a 0/1 "did the user listen at all" flag, which is the robust preference signal the book recommends for an implicit-feedback recommender.

</details>

**Problem 2.** A numeric feature "session_length" has no natural zero event and you still want a single binary split. How do you pick the threshold, and what is the trade-off?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Use the median of the feature as the threshold t. — _With no natural "did it happen" cutoff, the median splits the rows into a low half and a high half, so neither bin is empty._
- Apply b = 1[x >= median] to every row. — _This produces a balanced 0/1 feature: roughly half the rows are 1 and half are 0._
- Accept that you have discarded the magnitude. — _Binarizing throws away how far above or below the median each value is. If that magnitude is signal, prefer a log transform or binning instead._

**Answer:** Threshold at the median, giving a balanced 0/1 split (b = 1[x >= median]). The trade-off is that you discard all magnitude information — if the magnitude matters, use a log/power transform or binning rather than binarizing.

</details>